# Econ 3916 PS2 — The Survey That Wasn't Random

**ECON 3916 · Applied Data Analytics** · Topics 5-8 · 100 points · your role: Data Analyst, Growth team

This is the **starter**: the structure of the set in rubric order. Fill the empty cells; keep every heading so the grader finds each step. The brief on Canvas has the full text of every step, the numbers you must hit and the rubric. Every claim goes in a text cell: the reader is data analyst's audience, not a grader with a kernel.

**AI is not allowed in Phases 1–3.** Generative AI is **prohibited** in Phases 1 through 3. You write the simulations, the bootstrap loop and the tests yourself. AI is authorised and required in Phase 4. If you cannot explain a line you submitted, you cannot submit it.

## Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw reruns the same way
pd.set_option('display.precision', 3)

## Phase 1: Simulate the Population You Cannot See (30 pts)

### Step 1.1: Build a world with a known answer (15 pts)

In [2]:
# Parameters the brief fixes (Step 1.1). Keep them here so the grader can rerun you.
N_USERS = 50_000
BASE_BEFORE, BASE_AFTER = 3.432, 3.532      # true means 3.30 and 3.40

def simulate_population(base, n=N_USERS, rng=rng):
    """Return a DataFrame with engagement_minutes, engagement_z, had_support_ticket,
    latent s and the reported 1–5 satisfaction score."""
    engagement_minutes = rng.lognormal(mean=3.5, sigma=0.6, size=n)
    engagement_z = (engagement_minutes - engagement_minutes.mean())/ engagement_minutes.std()
    had_support_ticket = rng.random(n) < 0.22
    s = base + 0.35 * engagement_z - 0.60 * had_support_ticket + rng.normal(0, 0.75, size=n)
    satisfaction = np.clip(np.rint(s), 1, 5)
    return pd.DataFrame({
        "engagement_minutes": engagement_minutes,
        "engagement_z": engagement_z,
        "had_support_ticket": had_support_ticket,
        "s": s,
        "satisfaction": satisfaction,
    })
pop_before = simulate_population(BASE_BEFORE)
pop_after = simulate_population(BASE_AFTER)
true_before = pop_before["satisfaction"].mean()
true_after = pop_after["satisfaction"].mean()
print(f"true mean before {true_before:.3f} | after {true_after:.3f} | {true_after - true_before:+.3f}")





true mean before 3.291 | after 3.391 | +0.100


*Your answer here.*
I created a simulation that has 50,000 users before the redesign, and also has 50,000 after. It has a fixed seed, so it will return the same numbers every run. The satisfaction depends on how much time users spend, and if they have had a support ticket. The average satisfaction before was .3292, and after it was 3.39, so the redesign was marginally successful.
    

### Step 1.2: Impose the response mechanism (15 pts)

In [3]:
def respond(df, a, b, rng=rng):
    """Who answers the survey: P(respond) = logistic(a + b*(s - 3.3) + 0.30*engagement_z
    - 0.25*had_support_ticket); a user answers when a uniform draw is below it."""
    z = a + b * (df["s"] - 3.3) + .30 * df ["engagement_z"] - 0.25 * df ["had_support_ticket"]
    p = 1 / (1 + np.exp(-z))
    return rng.random(len(df)) < p

responded_before = respond(pop_before, a=-1.0, b=.45)
responded_after = respond(pop_after, a=-2.75, b=2.00)

obs_before = pop_before["satisfaction"][responded_before].mean()
obs_after = pop_after["satisfaction"][responded_after].mean()

print(f"response rate before{responded_before.mean():.1%} | after {responded_after.mean():.1%}")
print(f"observed before {obs_before:.3f} | after {obs_after:.3f}) | apparent gap {obs_after - obs_before:.3f}")
print(f"true before {true_before:.3f} | after {true_after:.3f} | true gap {true_after - true_before:.3f}")




response rate before27.5% | after 15.3%
observed before 3.592 | after 4.309) | apparent gap 0.717
true before 3.291 | after 3.391 | true gap 0.100


> **🛑 Verification checkpoint.** You should get an observed pre-redesign mean of
> about **3.60** (accept 3.55 to 3.65) and an observed post-redesign mean of
> about **4.31** (accept 4.25 to 4.35) — an apparent gap of roughly **0.71**
> points out of a true improvement of **0.10**. Response rates should fall from
> roughly 27% to roughly 15%. If your numbers are outside those bands, your
> response function or your `base` values are wrong. **Do not proceed until it
> matches** — everything after this depends on reproducing the PM's slide.

*Your answer here.*

This is due to bias, mostly non-response bias. After the redesign, the survery prompt targets very happy users, so the people who answer are typically going to be happier than the average user. That's why the survey overstates the jump in satisfaction.

This is not related to sample size, and the survey needs to adjust the question.

## Phase 2: Quantify What You Actually Know (35 pts)

### Step 2.1: Uncertainty without a formula (20 pts)

In [4]:
def bootstrap_by_hand(x, stat, B=2000, rng=rng):
    """Your own resampling loop; returns the B bootstrap statistics."""
    return np.array([stat(rng.choice(x, size=len(x), replace=True)) for _ in range(B)])

sample_before = pop_before["satisfaction"][responded_before].to_numpy()
sample_after = pop_after["satisfaction"][responded_after].to_numpy()

for label, sample in [("before", sample_before), ("after", sample_after)]:
  for name, stat in [("mean", np.mean), ("median", np.median)]:
    boot = bootstrap_by_hand(sample, stat)
    lo, hi = np.percentile(boot, [2.5, 97.5])
    pct = stats.bootstrap((sample,), stat, n_resamples=2_000, method = "percentile", batch=200, random_state=1).confidence_interval
    bca = stats.bootstrap((sample,), stat, n_resamples=2_000, method = "BCa", batch=200, random_state=1).confidence_interval
    print(f"{label} {name}: by hand [{lo:.3f}, {hi:.3f}] | scipy percentile [{pct.low:.3f}, {pct.high:.3f}] | BCa [{bca.low:.3f}, {bca.high:.3f}]")




before mean: by hand [3.578, 3.607] | scipy percentile [3.578, 3.608] | BCa [3.578, 3.608]


/usr/local/lib/python3.13/dist-packages/scipy/stats/_resampling.py:147: RuntimeWarning: invalid value encountered in scalar divide
  a_hat = 1/6 * sum(nums) / sum(dens)**(3/2)
/usr/local/lib/python3.13/dist-packages/scipy/_lib/_util.py:352: DegenerateDataWarning: The BCa confidence interval cannot be calculated. This problem is known to occur when the distribution is degenerate or the statistic is np.min.
  return fun(*args, **kwargs)


before median: by hand [4.000, 4.000] | scipy percentile [4.000, 4.000] | BCa [nan, nan]
after mean: by hand [4.294, 4.324] | scipy percentile [4.293, 4.325] | BCa [4.294, 4.325]
after median: by hand [4.000, 4.000] | scipy percentile [4.000, 4.000] | BCa [nan, nan]


I bootstrapped the mean and median of the respondants, with my own loop, and also using scipy stats.

The mean is about 3.59 before the redesign, and 4.31 after. The Percentile and BCa are almost identical for the mean.

They differ for the median, as it is 4 in both periods for percentile, and BCa can't be calculated as the values do not vary.

The intervals are tight but in the wrong rage, as the true mean is 3.39, which is very far off this figure.

### Step 2.2: Coverage is a claim you can check (15 pts)

In [5]:
n_reps, survey_size = 400, 2_000
hits_respondents = {"percentile": 0, "BCa": 0}
hits_everyone = {"percentile": 0, "BCa": 0}

for _ in range(n_reps):
  idx = rng.choice(N_USERS, size=survey_size, replace=False)
  survey = pop_after.iloc[idx]
  answered = respond(survey, a=-2.75, b=2.00).to_numpy()
  everyone = survey["satisfaction"].to_numpy()
  respondents = everyone[answered]
  for method in ["percentile", "BCa"]:
      ci = stats.bootstrap((respondents,), np.mean, n_resamples=1_000, method=method, random_state=rng).confidence_interval
      hits_respondents[method] += ci.low <= true_after <= ci.high
      ci = stats.bootstrap((everyone,), np.mean, n_resamples=1_000, method=method, random_state=rng).confidence_interval
      hits_everyone[method] += ci.low <= true_after <= ci.high

for method in ["percentile", "BCa"]:
  for label, hits in[("respondents only", hits_respondents), ("everyone answers", hits_everyone)]:
    cov = hits[method] / n_reps
    mc_se = np.sqrt(cov * (1 - cov) / n_reps)
    print(f"{method} | {label} | coverage {cov:.3f} +- {mc_se:.3f}")


percentile | respondents only | coverage 0.000 +- 0.000
percentile | everyone answers | coverage 0.945 +- 0.011
BCa | respondents only | coverage 0.000 +- 0.000
BCa | everyone answers | coverage 0.948 +- 0.011


I repeated the post- redesign survey 400 times. In every time,there were 2000 random users. The ones who answered were kept, and I built a percentile interval and BCa interval for their mean.

I counted how often the intervals contained the true mean, and it was 0.000. 0 of the 400 surveys had the true mean.

I built the same intervals as if every invited user answered, then the coverage was about 95% for both percentile and BCa. The gap between percentile and BCa was negligible.

The problem is not one of interval method, but the respondents of the survey. It is not representative of the users.

## Phase 3: Test Something Worth Testing (20 pts)

### Step 3.1: The test the PM wants

In [6]:
t_stat, p_value = stats.ttest_ind(sample_after, sample_before, equal_var=False)
diff = sample_after.mean() - sample_before.mean()
n1, n2 = len(sample_before), len(sample_after)
pooled_sd = np.sqrt(((n1 - 1) * sample_before.var(ddof=1) + (n2 - 1) * sample_after.var(ddof=1)) / (n1 + n2 - 2))
d = diff / pooled_sd
print(f"observed difference {diff:.3f} points | t = {t_stat:.1f} | p = {p_value:.2g} | Cohen's d = {d:.2f}")
print(f"true difference {true_after - true_before:.3f} points")

observed difference 0.717 points | t = 66.3 | p = 0 | Cohen's d = 0.88
true difference 0.100 points


I ran a two sample t test on the survey means.
The post redesign mean was .717 points higher, and t = 66.3 and p < .001. The effect size is large as cohens d was .88

The p-value tells us that a gap this big is unlikely to come from chance. The gap likely comes from who answered, and the true improvement is likely to be around .1 points. The test is focused on the wrong question.

### Step 3.2: Power, before the fact

In [7]:
from statsmodels.stats.power import TTestIndPower

sd = pop_before["satisfaction"].std(ddof=1)
d_target = 0.10 / sd
n_needed = TTestIndPower().solve_power(effect_size=d_target,  alpha=.05, power=.80)
n_get = len(sample_after)
d_detectable = TTestIndPower().solve_power(nobs1=n_get, alpha=.05, power=.80)
print(f"d for .10 points = {d_target:.3f} | n per group for 80% power = {n_needed:,.0f}")
print(f"with {n_get:,} per group: smallest detectable d = {d_detectable:.3f}, about {d_detectable * sd:.3f} points")

true_gap = true_after - true_before
for n_invited in [1_000, 5_000, 20_000, 50_000]:
  gaps = []
  for _ in range(200):
    survey_b = pop_before.iloc[rng.choice(N_USERS, size=n_invited, replace=False)]
    survey_a = pop_after.iloc[rng.choice(N_USERS, size=n_invited, replace=False)]
    ans_b = respond(survey_b, a=-1.0, b=.45).to_numpy()
    ans_a = respond(survey_a, a=-2.75, b=2.00).to_numpy()
    gap = survey_a["satisfaction"].to_numpy()[ans_a].mean() - survey_b["satisfaction"].to_numpy()[ans_b].mean()
    gaps.append(gap)
  gaps = np.array(gaps)
  print(f"invite {n_invited:,} | average estimated gap {gaps.mean():.3f} | spread {gaps.std(ddof=1):.3f} | bias {gaps.mean() - true_gap:+.3f}")


d for .10 points = 0.112 | n per group for 80% power = 1,249
with 7,638 per group: smallest detectable d = 0.045, about 0.040 points
invite 1,000 | average estimated gap 0.710 | spread 0.081 | bias +0.610
invite 5,000 | average estimated gap 0.709 | spread 0.033 | bias +0.609
invite 20,000 | average estimated gap 0.709 | spread 0.015 | bias +0.609
invite 50,000 | average estimated gap 0.708 | spread 0.009 | bias +0.609


A true improvement is going to be d = .10/.89, which = .112. To detect that number at 80% and an alpha of .05, we will need about 1250 users per group. The smallest effect detected with the 7638 responses post redesign was d = .045. The survey has a lot of power.

The power assumes a random sample, the survey simulated with the same response mechanism kept the gap at about .71 each time. This is .61 above the true gap of .10. The larger sample just makes the wrong answer even more confident, the real change has to come from who answers, and not more data.

## Phase 4: AI Expansion — The Co-Pilot Era (15 pts)

**Status: AI usage AUTHORISED and REQUIRED.** You have built the simulation, the bootstrap and the power analysis by hand. Now use the **P.R.I.M.E. framework** to go further than you could in the time.

### Task 4.1: The weighting attempt (10 pts)

In [8]:

import numpy as np
from sklearn.linear_model import LogisticRegression

# Generate the observed survey response indicators
responded = np.asarray(
    respond(pop_after, a=-2.75, b=2.00),
    dtype=bool
)

# Covariates are observed for all 50,000 users
X = pop_after[
    ["engagement_z", "had_support_ticket"]
].to_numpy(dtype=float)

# Satisfaction is used only for respondents in estimation
y = pop_after["satisfaction"].to_numpy(dtype=float)

def ipw_mean(X, y, responded):
    # Estimate P(response | observed covariates)
    model = LogisticRegression(
        C=1e6,
        max_iter=1000
    )
    model.fit(X, responded.astype(int))

    # Inverse response probabilities for respondents
    p_hat = model.predict_proba(X[responded])[:, 1]
    weights = 1.0 / p_hat

    # Normalized IPW / Hajek estimator
    return np.average(y[responded], weights=weights)


# Point estimates
naive_mean = y[responded].mean()
weighted_estimate = ipw_mean(X, y, responded)

# Bootstrap: resample all users and refit the response model
B = 1000
n = len(pop_after)
boot_estimates = []

for _ in range(B):
    idx = rng.integers(0, n, size=n)

    X_b = X[idx]
    y_b = y[idx]
    responded_b = responded[idx]

    if np.unique(responded_b).size < 2:
        continue

    boot_estimates.append(
        ipw_mean(X_b, y_b, responded_b)
    )

bootstrap_interval = tuple(
    np.percentile(boot_estimates, [2.5, 97.5])
)

print(f"True population mean:    {true_after:.4f}")
print(f"Naive survey mean:       {naive_mean:.4f}")
print(f"IPW weighted estimate:   {weighted_estimate:.4f}")
print(f"95% bootstrap interval:  {bootstrap_interval}")
print(f"Naive estimation error:  {naive_mean - true_after:+.4f}")
print(f"IPW estimation error:    {weighted_estimate - true_after:+.4f}")

# Oracle IPW: uses the TRUE response mechanism
# This is generally unavailable with real survey nonrespondents.

linear_predictor = (
    -2.75
    + 2.00 * (pop_after["s"].to_numpy() - 3.3)
    + 0.30 * pop_after["engagement_z"].to_numpy()
    - 0.25 * pop_after["had_support_ticket"].to_numpy()
)

p_true = 1 / (1 + np.exp(-linear_predictor))

oracle_estimate = np.average(
    y[responded],
    weights=1.0 / p_true[responded]
)

print(f"True mean:           {true_after:.4f}")
print(f"Covariate-only IPW:  {weighted_estimate:.4f}")
print(f"Oracle IPW:          {oracle_estimate:.4f}")


True population mean:    3.3909
Naive survey mean:       4.3099
IPW weighted estimate:   4.1516
95% bootstrap interval:  (np.float64(4.133006046327447), np.float64(4.1722025727149665))
Naive estimation error:  +0.9190
IPW estimation error:    +0.7607
True mean:           3.3909
Covariate-only IPW:  4.1516
Oracle IPW:          3.4007


*Your answer here.*

Revision: Version 1 did not use my variable names, instead made up a frame called "df." It caused failure with a NameError. I revised it by inserting my variable names so it could run on my data.

Check : Among people who answered, the mean after the design is 4.31. The weighted estimate is 4.15, and the bootstrap interval is from 4.13 to 4.17. The true mean is 3.39, so this means correction didnt work, as the interval is still far from the true mean, which was the expected result.

Ownership: The model is fit on all 50,000 users, using both engagement and support tickets  to predict who would answer. (model.fit).predict_proba, gives each users chance of answering, and the people get a weight of 1 divided by that chance, so it factors in people who were unlikely to answer.

I would use AI for this again, because it was fast with the right variables once given, and went beyond my technical means.

### Task 4.2: The slide (5 pts)


The post-redesign satisfaction is complex, and there are things we can and cannot say for sure. The survey is unable to tell us whether or not the redesign actually improved satisfaction.

It tells us, that among users who answered after the redesign, average satisfaction was 4.3 out of 5, with a 95% interval from 4.29 to 4.32. However, that number describes the people who answered and not all users.

The survey misleadingly displayed that the satisfaction rose from 3.6 to 4.3, as  it did not show that the new prompt changed who answers. The response rate fell from 27 to 15%, and it was reframed so the users who answer are likely to be happy.

I would change the data collection, not by increasing the size, but the randomness of collection. The users surveyed should be random, and the ones who don't answer should be followed up with to create a more complete vision of the users.

### AI Transcript

Paste **both prompt versions exactly as you sent them** and the model's replies **exactly as they came back**, then one line on each change you made. Name the model and the date.

*Model and date:*
10/26 - GPT-6

*Version 1 prompt:*
```csharp
[Prep] Act as a survey statistician who works on non-response
adjustment for product research.

[Request] I have a simulated user population with satisfaction scores
and two covariates, engagement_minutes and had_support_ticket, plus a
non-response mechanism where the probability of responding increases
with satisfaction. Write Python that implements inverse-probability
weighting to correct the observed survey mean, using a logistic model
of response on the covariates.

[Iterate] numpy, pandas, scikit-learn or statsmodels. Reuse my
variable names. Return the weighted estimate and its bootstrap
interval.

[Mechanism Check] Explain precisely which assumption makes this
correction valid, and what happens to it when response depends on
satisfaction ITSELF rather than only on the covariates.

[Evaluate] Given my mechanism, will this correction recover the true
mean? Answer yes or no and justify it.
```

*Reply:*
The key result: No. Given your non-response mechanism, inverse-probability weighting (IPW) using only `engagement_minutes` and `had_support_ticket` will not generally recover the true population satisfaction mean. It can reduce bias, but it cannot eliminate bias caused by satisfaction itself influencing response probability after conditioning on those covariates.

Here's a Python implementation that lets you measure the remaining bias in your simulated population.

## 1. Python implementation

The code assumes your DataFrame `df` has these columns:

- `satisfaction`: satisfaction score (missing for nonrespondents in real survey data)
- `engagement_minutes`: observed for everyone
- `had_support_ticket`: observed for everyone
- `responded`: 1 if the user answered the survey, 0 otherwise

It estimates response probabilities using logistic regression, calculates a normalized IPW (Hájek) mean, and returns a 95% bootstrap confidence interval. Each bootstrap iteration refits the response model.

```
import numpy as npimport pandas as pdfrom sklearn.linear_model import LogisticRegressionfrom sklearn.pipeline import make_pipelinefrom sklearn.preprocessing import StandardScalercovariates = ["engagement_minutes", "had_support_ticket"]def ipw_mean(data):    X = data[covariates]    R = data["responded"].astype(bool).to_numpy()    # Fit response model on respondents AND nonrespondents    model = make_pipeline(        StandardScaler(),        LogisticRegression(max_iter=2000)    )    model.fit(X, R.astype(int))    # Estimated P(responded = 1 | covariates)    p_response = model.predict_proba(X)[:, 1]    # Only respondents contribute satisfaction outcomes    weights = 1.0 / p_response[R]    scores = data.loc[R, "satisfaction"].to_numpy()    # Normalized inverse-probability weighted mean    return np.sum(weights * scores) / np.sum(weights)def bootstrap_ipw(df, B=2000, seed=42):    rng = np.random.default_rng(seed)    n = len(df)    estimate = ipw_mean(df)    boot_estimates = []    for _ in range(B):        indices = rng.integers(0, n, size=n)        sample = df.iloc[indices].reset_index(drop=True)        if sample["responded"].nunique() < 2:            continue        boot_estimates.append(ipw_mean(sample))    ci = np.percentile(boot_estimates, [2.5, 97.5])    return estimate, tuple(ci)# Survey estimatesnaive_mean = df.loc[df["responded"] == 1, "satisfaction"].mean()weighted_estimate, bootstrap_interval = bootstrap_ipw(df)print(f"Naive respondent mean: {naive_mean:.3f}")print(f"IPW estimate: {weighted_estimate:.3f}")
```

The interval quantifies sampling uncertainty under a row-bootstrap approximation; it does not account for bias arising from an incorrect response mechanism.

## 2. The IPW estimator

Let \\(R_i\\) indicate survey response, \\(Y_i\\) denote satisfaction, and \\(X_i\\) contain the two observed covariates.

Estimate each user's response propensity:

\\[ \hat p_i=P(R_i=1\mid X_i) \\]

Assign each respondent a weight:

\\[ w_i=\frac{1}{\hat p_i} \\]

Then estimate population satisfaction using:

\\[ \boxed{ \hat\mu\_{\mathrm{IPW}}= \frac{\sum\_{i=1}^n R_iw_iY_i} {\sum\_{i=1}^n R_iw_i} } \\]

Users with low predicted response probabilities receive larger weights because they represent more people who did not respond.

## 3. Mechanism check: the critical assumption

IPW requires Missing at Random (MAR), conditional on the observed covariates:

\\[ \boxed{R\perp Y\mid X} \\]

Equivalently:

\\[ P(R=1\mid X,Y)=P(R=1\mid X) \\]

This means that once engagement minutes and support-ticket status are known, satisfaction must provide no additional information about the likelihood of responding.

Valid correction also requires positive response probability for all relevant covariate profiles and a correctly specified response propensity model.

Your simulation instead specifies:

\\[ P(R=1\mid X,Y)=f(X,Y) \\]

where response probability increases with satisfaction itself. If that relationship remains after conditioning on \\(X\\), the MAR assumption is violated. This is Missing Not at Random (MNAR).

## 4. What happens to the estimates?

Naive survey mean

## Upward bias

Satisfied users respond more often, making the observed respondents unrepresentative.

Covariate-only IPW

## Residual bias

Corrects selection associated with observed covariates, but not remaining satisfaction-dependent selection.

For intuition, suppose high-satisfaction users are twice as likely to respond as low-satisfaction users, even when both have identical engagement and support-ticket histories.

A logistic model using only those histories cannot distinguish their response probabilities. Both receive approximately the same weight, so high-satisfaction users remain overrepresented.

The IPW estimate may therefore remain upward biased. Its confidence interval may be narrow while still missing the true population mean.

## 5. Evaluation

Will this correction recover the true mean? No, not reliably.

It would work under MAR, positivity, and correct propensity modeling. Your stated mechanism violates MAR when satisfaction independently affects response.

Because you have simulated data, you can directly compare the true population mean with both the naive estimate and the covariate-adjusted IPW estimate to quantify the failure.

One important distinction: if the simulation exposes the true response probabilities, including their dependence on satisfaction, an IPW estimator based on those true probabilities can recover the population mean in expectation. But in an actual survey, satisfaction is unobserved for nonrespondents, so that response model cannot generally be identified from survey data alone.

For your lab conclusion: Covariate-based inverse-probability weighting adjusts for selection on observed characteristics, but cannot generally remove non-response bias driven by the missing outcome itself. A bootstrap confidence interval measures uncertainty around the estimator, not whether the estimator is unbiased.

*Version 2 prompt:*





```
[Prep] Act as a survey statistician who works on non-response
adjustment for product research.

[Request] I have a simulated user population with satisfaction scores
and two covariates, engagement_minutes and had_support_ticket, plus a
non-response mechanism where the probability of responding increases
with satisfaction. Write Python that implements inverse-probability
weighting to correct the observed survey mean, using a logistic model
of response on the covariates.

My data: pop_after is a pandas DataFrame with 50,000 users and columns
engagement_minutes, engagement_z, had_support_ticket, s and
satisfaction. There is no "responded" column. respond(pop_after,
a=-2.75, b=2.00) returns a True/False Series for who answers, using
P(respond) = logistic(a + b*(s - 3.3) + 0.30*engagement_z
- 0.25*had_support_ticket). Correct the post-redesign survey mean and
compare it to true_after, the true population mean. Use my rng (a
NumPy Generator) for anything random.

[Iterate] numpy, pandas, scikit-learn or statsmodels. Reuse my
variable names. Return the weighted estimate and its bootstrap
interval.

[Mechanism Check] Explain precisely which assumption makes this
correction valid, and what happens to it when response depends on
satisfaction ITSELF rather than only on the covariates.

[Evaluate] Given my mechanism, will this correction recover the true
mean? Answer yes or no and justify it.

```

*Reply:*
Answer: No. With your specified response mechanism, inverse-probability weighting (IPW) based only on `engagement_z` and `had_support_ticket` will not generally recover `true_after`. It may reduce non-response bias, but satisfaction directly affects the probability of responding, violating the key identification assumption.

## 1. Python implementation

This code uses your existing `pop_after`, `respond()`, `true_after`, and `rng` variables. It fits a logistic response model using only the two covariates, calculates the normalized IPW estimate, and constructs a 95% bootstrap confidence interval.

`engagement_z` is used as the standardized version of `engagement_minutes`, matching your simulated response mechanism.

```
import numpy as npfrom sklearn.linear_model import LogisticRegression# Generate the observed survey response indicatorsresponded = np.asarray(    respond(pop_after, a=-2.75, b=2.00),    dtype=bool)# Covariates are observed for all 50,000 usersX = pop_after[    ["engagement_z", "had_support_ticket"]].to_numpy(dtype=float)# Satisfaction is used only for respondents in estimationy = pop_after["satisfaction"].to_numpy(dtype=float)def ipw_mean(X, y, responded):    # Estimate P(response | observed covariates)    model = LogisticRegression(        C=1e6,        max_iter=1000    )    model.fit(X, responded.astype(int))    # Inverse response probabilities for respondents    p_hat = model.predict_proba(X[responded])[:, 1]    weights = 1.0 / p_hat    # Normalized IPW / Hajek estimator    return np.average(y[responded], weights=weights)# Point estimatesnaive_mean = y[responded].mean()weighted_estimate = ipw_mean(X, y, responded)# Bootstrap: resample all users and refit the response modelB = 1000n = len(pop_after)boot_estimates = []for _ in range(B):    idx = rng.integers(0, n, size=n)    X_b = X[idx]    y_b = y[idx]    responded_b = responded[idx]    if np.unique(responded_b).size < 2:        continue    boot_estimates.append(        ipw_mean(X_b, y_b, responded_b)    )bootstrap_interval = tuple(    np.percentile(boot_estimates, [2.5, 97.5]))print(f"True population mean:    {true_after:.4f}")print(f"Naive survey mean:       {naive_mean:.4f}")
```

The bootstrap resamples the full simulated population, including respondents and nonrespondents, and refits the logistic model each iteration. This incorporates response-model estimation uncertainty into the percentile interval.

## 2. Mechanism check

The crucial assumption is Missing at Random (MAR) conditional on the covariates:

\\[ R\perp Y\mid X \\]

Equivalently,

\\[ P(R=1\mid X,Y)=P(R=1\mid X) \\]

where \\(R\\) is the response indicator, \\(Y\\) is satisfaction, and \\(X\\) contains the observed covariates.

IPW additionally requires positivity, \\(P(R=1\mid X)>0\\), and a correctly specified propensity model.

Your actual response mechanism is:

\\[ P(R=1)=\operatorname{logistic}\left( -2.75+2.00(s-3.3)+0.30\\,engagement_z -0.25\\,had\\\_support\\\_ticket \right) \\]

Because the probability of responding increases directly with satisfaction (`s`), the MAR assumption fails if this dependence persists after controlling for the covariates.

This is Missing Not at Random (MNAR).

Consequently, even perfectly estimating \\(P(R=1\mid X)\\) cannot remove the residual selection bias. The covariate-only IPW estimator generally converges to

\\[ E_X[E(Y\mid X,R=1)] \\]

rather than the target

\\[ E_X[E(Y\mid X)]. \\]

## 3. Evaluation: will IPW recover the true mean?

No, not consistently under your stated mechanism.

Users with higher satisfaction are more likely to respond, even among users with similar engagement and support-ticket histories. The logistic model cannot account for that remaining difference because it excludes satisfaction.

If `s` is satisfaction itself, or positively related to it after conditioning on the covariates, the IPW estimate will generally remain upward biased.

One useful simulation-only check is to calculate IPW using the actual response probabilities, which your simulation makes available:

```
# Oracle IPW: uses the TRUE response mechanism# This is generally unavailable with real survey nonrespondents.linear_predictor = (    -2.75    + 2.00 * (pop_after["s"].to_numpy() - 3.3)    + 0.30 * pop_after["engagement_z"].to_numpy()    - 0.25 * pop_after["had_support_ticket"].to_numpy())p_true = 1 / (1 + np.exp(-linear_predictor))oracle_estimate = np.average(    y[responded],    weights=1.0 / p_true[responded])print(f"True mean:           {true_after:.4f}")print(f"Covariate-only IPW:  {weighted_estimate:.4f}")print(f"Oracle IPW:          {oracle_estimate:.4f}")
```

Oracle IPW should approach the true mean with a sufficiently large population and appropriate response probabilities, because it accounts for the actual selection mechanism. It need not match exactly in a finite sample.

Conclusion for your lab: Covariate-based IPW can correct non-response bias when response is conditionally independent of satisfaction given the observed covariates. Your mechanism violates this assumption because satisfaction directly influences response. Therefore, the adjusted mean may improve on the naive survey mean but is not guaranteed to recover `true_after`. The bootstrap interval captures sampling variability, not the remaining MNAR bias.

## Before you submit

- [ ] Every heading above is still here, in order, with work under it.
- [ ] Every number you report is produced by a cell above it, and every claim has a text cell.
- [ ] The verification checkpoint(s) are met before anything built on them.
- [ ] Phase 4: the AI transcript is the exchange, not a summary; both prompt versions are in it.
- [ ] Submit the Colab share link on Canvas (upload the `.ipynb` as a fallback).
- [ ] GitHub portfolio: *File › Save a copy in GitHub*, path `Problem Set 2/Econ_3916_PS2.ipynb`, with a short `README.md` in that folder.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.